# Restaurant Sales Intelligence

## Menu Performance, Peak Hours, and Revenue Insights

This notebook explores the restaurant sales dataset from a business perspective. The focus is on menu mix, time-of-day demand, monthly revenue patterns, payment mix, and the data quality issues that affect reporting confidence.


## Analysis goals

- Quantify revenue, orders, and average order value.
- Identify the highest-performing items by revenue and volume.
- Compare sales patterns by item type, time of sale, and payment method.
- Highlight data quality limits that matter for business interpretation.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.append(str(project_root / "src"))

from analysis_pipeline import (  # noqa: E402
    TIME_OF_SALE_ORDER,
    build_data_quality_summary,
    build_summary_tables,
    load_raw_data,
    prepare_clean_dataset,
)

pd.options.display.float_format = "{:,.2f}".format
plt.style.use("ggplot")


In [2]:
raw_df = load_raw_data(project_root / "data" / "raw" / "restaurant_sales.csv")
clean_df = prepare_clean_dataset(raw_df)
quality_df = build_data_quality_summary(clean_df)
tables = build_summary_tables(clean_df)

print(f"Rows: {len(clean_df):,}")
print(f"Date range: {clean_df['order_date'].min()} to {clean_df['order_date'].max()}")


Rows: 1,000
Date range: 2022-01-04 00:00:00 to 2023-12-03 00:00:00


## Data quality review

The source file is usable, but a few issues should be documented before interpreting results:

- Dates appear in two different formats and need explicit parsing logic.
- Payment method is missing for a subset of transactions.
- `received_by` only contains `Mr.` and `Mrs.`, so it should not be treated as an employee identifier.
- Transaction amounts are internally consistent with unit price multiplied by quantity.


In [3]:
display(quality_df)

clean_df[
    [
        "missing_transaction_type_flag",
        "parsed_date_missing_flag",
        "duplicate_order_id_flag",
        "transaction_amount_matches_expected",
    ]
].agg(["sum"]).T.rename(columns={"sum": "count"})


,issue_type,affected_rows,status,detail
0,mixed_date_formats,1000,reviewed,403 rows used dd-mm-yyyy and 597 rows used mm/...
1,missing_transaction_type,107,handled,Missing payment values are retained as Unknown...
2,transaction_amount_validation,0,validated,Validated whether transaction_amount equals it...
3,duplicate_order_id_check,0,validated,Checked whether order_id values repeat across ...
4,received_by_field_limitation,1000,documented,received_by contains only honorific labels ['M...
5,unparsed_dates,0,validated,Any rows listed here could not be parsed into ...


,count
missing_transaction_type_flag,107
parsed_date_missing_flag,0
duplicate_order_id_flag,0
transaction_amount_matches_expected,1000


## KPI snapshot

These KPIs summarize the scale of the business captured in the dataset.


In [4]:
display(tables["kpi_summary"])


,metric,value
0,total_revenue,"275,230.00"
1,total_orders,"1,000.00"
2,total_quantity_sold,"8,162.00"
3,average_order_value,275.23
4,average_quantity_per_order,8.16
5,missing_payment_share_pct,10.70


## Product performance

Revenue and unit volume do not tell exactly the same story. A product can lead in revenue because of a higher selling price, while another product can lead in quantity because of stronger unit demand.


In [5]:
display(tables["revenue_by_item"][["item_name", "revenue", "revenue_share_pct", "orders", "average_order_value"]])
display(tables["quantity_by_item"][["item_name", "quantity_sold", "quantity_share_pct", "orders"]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rev = tables["revenue_by_item"].sort_values("revenue", ascending=True)
qty = tables["quantity_by_item"].sort_values("quantity_sold", ascending=True)

axes[0].barh(rev["item_name"], rev["revenue"], color="#0b6e4f")
axes[0].set_title("Revenue by Item")
axes[0].set_xlabel("Revenue")

axes[1].barh(qty["item_name"], qty["quantity_sold"], color="#c75c00")
axes[1].set_title("Quantity Sold by Item")
axes[1].set_xlabel("Units Sold")

plt.tight_layout()
plt.show()


,item_name,revenue,revenue_share_pct,orders,average_order_value
4,Sandwich,65820,23.91,129,510.23
2,Frankie,57500,20.89,139,413.67
1,Cold coffee,54440,19.78,161,338.14
5,Sugarcane juice,31950,11.61,153,208.82
3,Panipuri,24520,8.91,150,163.47
0,Aalopuri,20880,7.59,134,155.82
6,Vadapav,20120,7.31,134,150.15


,item_name,quantity_sold,quantity_share_pct,orders
0,Cold coffee,1361,16.67,161
1,Sugarcane juice,1278,15.66,153
2,Panipuri,1226,15.02,150
3,Frankie,1150,14.09,139
4,Sandwich,1097,13.44,129
5,Aalopuri,1044,12.79,134
6,Vadapav,1006,12.33,134


C:\Users\sivar\AppData\Local\Temp\ipykernel_11808\712467278.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Demand patterns by daypart and month

Daypart analysis helps with staffing, prep timing, and promotions. Monthly trends help show whether demand is consistent or uneven over time.


In [6]:
display(tables["sales_by_time_of_sale"])
display(tables["monthly_sales_trend"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

time_table = tables["sales_by_time_of_sale"].set_index("time_of_sale").reindex(TIME_OF_SALE_ORDER).reset_index()
axes[0].bar(time_table["time_of_sale"], time_table["revenue"], color="#4c7cbf")
axes[0].set_title("Revenue by Time of Sale")
axes[0].tick_params(axis="x", rotation=20)

month_table = tables["monthly_sales_trend"]
axes[1].plot(month_table["order_month"], month_table["revenue"], marker="o", color="#0b6e4f")
axes[1].set_title("Monthly Revenue Trend")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


,time_of_sale,orders,quantity_sold,revenue,average_order_value,revenue_share_pct
0,Morning,190,1574,53730,282.79,19.52
1,Afternoon,205,1714,56345,274.85,20.47
2,Evening,201,1540,52355,260.47,19.02
3,Night,205,1759,62075,302.80,22.55
4,Midnight,199,1575,50725,254.90,18.43


,order_month,orders,quantity_sold,revenue,average_order_value
0,2022-01,21,165,5785,275.48
1,2022-02,27,206,8010,296.67
2,2022-03,23,161,4720,205.22
3,2022-04,65,509,17610,270.92
4,2022-05,84,658,21155,251.85
5,2022-06,66,485,16670,252.58
6,2022-07,66,513,17420,263.94
7,2022-08,85,710,21285,250.41
8,2022-09,82,641,21340,260.24
9,2022-10,76,591,19240,253.16


C:\Users\sivar\AppData\Local\Temp\ipykernel_11808\2475458451.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Payment mix and category mix

Payment mix is useful for checkout analysis, but the missing values need to stay visible. Item type mix shows whether the business is more dependent on fast food or beverages.


In [7]:
display(tables["sales_by_payment_type"])
display(tables["item_type_mix"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

payment_table = tables["sales_by_payment_type"]
axes[0].bar(payment_table["payment_type"], payment_table["revenue"], color="#c75c00")
axes[0].set_title("Revenue by Payment Type")

item_type_table = tables["item_type_mix"]
axes[1].bar(item_type_table["item_type"], item_type_table["revenue_share_pct"], color="#0b6e4f")
axes[1].set_title("Item Type Revenue Share")
axes[1].set_ylabel("Share (%)")

plt.tight_layout()
plt.show()


,payment_type,orders,quantity_sold,revenue,order_share_pct,average_order_value
0,Cash,476,3943,132840,47.60,279.08
1,Online,417,3291,110595,41.70,265.22
2,Unknown,107,928,31795,10.70,297.15


,item_type,orders,quantity_sold,revenue,revenue_share_pct,quantity_share_pct
1,Fastfood,686,5523,188840,68.61,67.67
0,Beverages,314,2639,86390,31.39,32.33


C:\Users\sivar\AppData\Local\Temp\ipykernel_11808\1437339524.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Combined performance view

Looking at item type and time of sale together helps identify where category-specific operations or promotions could be most effective.


In [8]:
display(tables["item_type_time_of_sale_performance"])


,item_type,time_of_sale,orders,quantity_sold,revenue,average_order_value
0,Beverages,Morning,57,493,16630,291.75
1,Beverages,Afternoon,75,662,21605,288.07
2,Beverages,Evening,62,518,16475,265.73
3,Beverages,Night,66,526,17635,267.20
4,Beverages,Midnight,54,440,14045,260.09
5,Fastfood,Morning,133,1081,37100,278.95
6,Fastfood,Afternoon,130,1052,34740,267.23
7,Fastfood,Evening,139,1022,35880,258.13
8,Fastfood,Night,139,1233,44440,319.71
9,Fastfood,Midnight,145,1135,36680,252.97


## Business interpretation

- Sandwich leads revenue, which makes it a strong anchor product for bundles and featured placement.
- Cold coffee leads volume, which suggests beverages can support traffic and add-on selling even when they do not top revenue.
- Night is the strongest overall daypart, while beverage demand is relatively strongest in the afternoon.
- Late-2023 monthly totals are much lower, but the underlying record counts are also sparse, so coverage limitations must be acknowledged before drawing trend conclusions.
